In [9]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from medpy.metric.binary import hd95
import warnings
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# 忽略 MedPy 的 RuntimeWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ================= 配置区 =================
SAM2_CFG_NAME = "configs/sam2.1/sam2.1_hiera_b+"

# 【重要】必须指向官方原始权重的路径
BASE_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/busi_bLoss_adapter_run18/checkpoints/checkpoint.pt" 

#busi_fewshot_10_run1
# 少样本权重的路径字典
CHECKPOINTS = {
    "10%": "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/medsam2_baseline_10_run1/checkpoints/checkpoint.pt",
    "25%": "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/medsam2_baseline_25_run1/checkpoints/checkpoint.pt",
    "50%": "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/medsam2_baseline_50_run1/checkpoints/checkpoint.pt"
}

# 数据集设置 (注意确认是 BUSI 还是 Kvasir)
SPLIT_DATASET_ROOT = "/home/zhengsongming/jupyterworkspace/datasets/BUSI_for_SAM2"
TEST_SET_FILE = os.path.join(SPLIT_DATASET_ROOT, "ImageSets", "val.txt")
IMAGES_DIR = os.path.join(SPLIT_DATASET_ROOT, "JPEGImages")
ANNOTATIONS_DIR = os.path.join(SPLIT_DATASET_ROOT, "Annotations")

HYDRA_OVERRIDES = [
    "model.image_encoder.trunk._target_=sam2.modeling.backbones.hieradet_adapterv1.Hiera",
    "+model.image_encoder.trunk.use_adapter=True",
    "+model.use_adapter=True",
]

# ================= 核心函数：三明治加载 (解决 0.36% 问题) =================
def load_model_sandwich(cfg_path, base_ckpt, finetuned_ckpt, overrides):
    print(f"\n>>> [加载] Base: {os.path.basename(base_ckpt)} + Adapter: {os.path.basename(finetuned_ckpt)}")
    
    # 1. 加载官方 Base 权重 (确保主干正常)
    if not os.path.exists(base_ckpt):
        raise FileNotFoundError(f"找不到官方权重: {base_ckpt}")
    model = build_sam2(cfg_path, base_ckpt, device='cuda', hydra_overrides_extra=overrides)
    
    # 2. 覆盖微调的 Adapter 权重
    if os.path.exists(finetuned_ckpt):
        ft_ckpt = torch.load(finetuned_ckpt, map_location='cuda')
        state_dict = ft_ckpt["model"] if "model" in ft_ckpt else ft_ckpt
        
        new_state_dict = {}
        for k, v in state_dict.items():
            k = k.replace("model.", "") # 去掉 DDP 前缀
            new_state_dict[k] = v
        
        # 宽容加载 (只覆盖存在的 Adapter 参数)
        model.load_state_dict(new_state_dict, strict=False)
        print(">>> 微调参数覆盖完成。")
    else:
        print(f">>> 警告: 找不到微调权重 {finetuned_ckpt}，将仅使用 Base 模型评估！")

    return model

# ================= 指标计算 (包含 HD95) =================
def calculate_metrics(gt_mask, pred_mask):
    gt_bool = gt_mask > 0
    pred_bool = pred_mask > 0
    
    # Dice & IoU
    intersection = np.logical_and(gt_bool, pred_bool).sum()
    dice = (2. * intersection) / (gt_bool.sum() + pred_bool.sum() + 1e-8)
    union = gt_bool.sum() + pred_bool.sum() - intersection
    iou = intersection / (union + 1e-8)
    
    # HD95
    if pred_bool.sum() == 0 or gt_bool.sum() == 0:
        hd95_score = np.nan
    else:
        try:
            hd95_score = hd95(pred_bool, gt_bool)
        except Exception:
            hd95_score = np.nan
            
    return dice, iou, hd95_score

# ================= 主评估循环 =================
def eval_one_ratio(ratio, ckpt_path):
    # 使用三明治加载
    try:
        model = load_model_sandwich(SAM2_CFG_NAME, BASE_CHECKPOINT_PATH, ckpt_path, HYDRA_OVERRIDES)
    except Exception as e:
        print(f"Error loading {ratio}: {e}")
        return None

    predictor = SAM2ImagePredictor(model)
    
    with open(TEST_SET_FILE, 'r') as f:
        samples = [line.strip() for line in f.readlines()]
        
    results = []
    for sample in tqdm(samples, desc=f"Eval {ratio}", leave=False):
        img_p = os.path.join(IMAGES_DIR, sample, "00000.jpg")
        mask_p = os.path.join(ANNOTATIONS_DIR, sample, "00000.png")
        if not os.path.exists(img_p): continue
        
        image = cv2.imread(img_p)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        gt_mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
        
        # Box Prompt
        contours, _ = cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: continue
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        
        predictor.set_image(image_rgb)
        masks, _, _ = predictor.predict(box=np.array([[x, y, x + w, y + h]]), multimask_output=False)
        pred_mask = (masks[0] * 255).astype(np.uint8)
        
        # 计算指标
        d, i, h95 = calculate_metrics(gt_mask, pred_mask)
        results.append({"dice": d, "iou": i, "hd95": h95})
        predictor.reset_predictor()
        
    # 释放显存
    del model, predictor
    torch.cuda.empty_cache()
    
    # 返回平均值
    df = pd.DataFrame(results)
    return {
        "Dice": df['dice'].mean() * 100,
        "IoU": df['iou'].mean() * 100,
        "HD95": df['hd95'].mean()
    }

# ================= 执行 =================
if __name__ == "__main__":
    final_table = []
    print("Start Evaluation...")
    
    for ratio, ckpt in CHECKPOINTS.items():
        res = eval_one_ratio(ratio, ckpt)
        if res:
            res["Ratio"] = ratio
            final_table.append(res)
            
    print("\n" + "="*50)
    print(f"{'Ratio':<10} | {'Dice(%)':<10} | {'IoU(%)':<10} | {'HD95':<10}")
    print("-" * 50)
    for row in final_table:
        print(f"{row['Ratio']:<10} | {row['Dice']:<10.2f} | {row['IoU']:<10.2f} | {row['HD95']:<10.2f}")
    print("="*50)

Start Evaluation...

>>> [加载] Base: sam2.1_hiera_tiny_finetune512.yaml + Adapter: checkpoint.pt
Error loading 10%: Weights only load failed. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: Unsupported operand 35

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

>>> [加载] Base: sam2.1_hiera_tiny_finetune512.yaml + Adapter: checkpoint.pt
Error loading 25%: Weights only load failed. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can m